# Data Processing

In [1]:
import os
import sys
import torch
import cv2
import numpy as np
import pandas as pd
from ultralytics import YOLO
from facial_emotion_recognition import EmotionRecognition
import mediapipe as mp
from tqdm import tqdm
import logging
import pympi
import gc

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
device = "cuda" if torch.cuda.is_available() else "cpu"
logging.info(f"Using device: {device}")

2025-12-10 12:30:11,393 [INFO] Using device: cuda


In [2]:
def create_labels_from_filenames(root_dir):
    labels_dict = {}
    skipped_files = []
    
    print(f"📂 Scanning {root_dir} for labels...")
    
    for root, dirs, files in os.walk(root_dir):
        for file in files:
            if not file.lower().endswith(('.mp4', '.avi', '.mov', '.mkv')):
                continue
            
            name_lower = file.lower()
            
            if 'lie' in name_lower:
                labels_dict[file] = 1
            elif 'truth' in name_lower:
                labels_dict[file] = 0
            else:
                skipped_files.append(file)

    print(f"✅ Found {len(labels_dict)} labeled videos.")
    if skipped_files:
        print(f"⚠️ Warning: Could not determine label for {len(skipped_files)} videos (e.g., {skipped_files[:3]}).")
        
    return labels_dict

# Segmenting and .eaf parsing

In [3]:
import abc

class BaseSegmenter(abc.ABC):
    @abc.abstractmethod
    def get_segments(self, video_path):
        pass

class SilesianSegmenter(BaseSegmenter):
    def __init__(self, fps=100):
        self.fps = fps

    def _convert_timestamp(self, timestamp_ms):
        return int((timestamp_ms / 1000.0) * self.fps)

    def get_segments(self, video_path):
        eaf_path = video_path.replace('.avi', '.eaf')
        if not os.path.exists(eaf_path):
            logging.warning(f"Annotation file missing: {eaf_path}")
            return []
        try:
            eaf = pympi.Elan.Eaf(eaf_path)
            annotations = eaf.get_annotation_data_for_tier('Question')
        except Exception as e:
            logging.error(f"Failed to parse EAF {eaf_path}: {e}")
            return []
        
        segments = []
        for i, (start, end, value) in enumerate(annotations):
            if value == 'Correct':
                is_deceptive = 1 if (i not in [0, 1, 8]) else 0 
                
                segments.append((
                    self._convert_timestamp(start), 
                    self._convert_timestamp(end), 
                    is_deceptive
                ))
        return segments

class SimpleLabelSegmenter(BaseSegmenter):
    """
    For datasets where 1 video = 1 label.
    Expects a dictionary mapping filenames to labels.
    """
    def __init__(self, label_map, video_fps=30):
        self.label_map = label_map
        self.fps = video_fps

    def get_segments(self, video_path):
        filename = os.path.basename(video_path)
        if filename not in self.label_map:
            return []
        
        label = self.label_map[filename]
        
        cap = cv2.VideoCapture(video_path)
        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        cap.release()
        
        return [(0, total_frames, label)]

### Face detection and crop (YOLO)

In [4]:
def detect_faces(model, frame):
    results = model(frame, verbose=False)
    if not results or results[0].boxes is None:
        return []
    return results[0].boxes.xyxy.int().tolist()

def face_crop(model, frame):
    boxes = detect_faces(model, frame)

    for _, box in enumerate(boxes):
        x1, y1, x2, y2 = map(int, box)
        
        h, w = frame.shape[:2]
        x1, y1 = max(0, x1), max(0, y1)
        x2, y2 = min(w, x2), min(h, y2)
        face_crop = frame[y1:y2, x1:x2]
        if face_crop.size == 0:
            continue

        return face_crop, (x1, y1, x2, y2)
    
    return None, None

### Resize images to consistent size

In [5]:
def resize_frame(frame, size=(224, 224)):
    return cv2.resize(frame, size)

### Geometric face normalization with MediaPipe

In [6]:
def geometric_normalization(frame, landmarks):
    if not landmarks:
        return frame

    LEFT_EYE_LANDMARKS = [33, 133]
    RIGHT_EYE_LANDMARKS = [362, 263]

    h, w, _ = frame.shape
    
    left_eye = np.array([[landmarks[i].x * w, landmarks[i].y * h] for i in LEFT_EYE_LANDMARKS]).mean(axis=0)
    right_eye = np.array([[landmarks[i].x * w, landmarks[i].y * h] for i in RIGHT_EYE_LANDMARKS]).mean(axis=0)

    dy = right_eye[1] - left_eye[1]
    dx = right_eye[0] - left_eye[0]
    angle = np.degrees(np.arctan2(dy, dx))

    center = tuple(map(float, np.mean([left_eye, right_eye], axis=0)))
    rot_mat = cv2.getRotationMatrix2D(center, angle, 1.0)
    aligned = cv2.warpAffine(frame, rot_mat, (w, h), flags=cv2.INTER_CUBIC)

    return aligned

### Emotion Detection

In [7]:
def get_emotion_probs(frame, emotion_detector):
    if frame.ndim == 3:
        frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    tensor = emotion_detector.transform(frame).unsqueeze(0).to(emotion_detector.device)

    with torch.no_grad():
        output = emotion_detector.network(tensor)
        probs = torch.softmax(output, dim=1).cpu().numpy()[0]

    return {emotion_detector.emotions[i]: float(probs[i]) for i in range(len(probs))}

def detect_emotions(frame, emotion_detector):
    return get_emotion_probs(frame, emotion_detector)

### Face Landmarks

In [8]:
def extract_landmarks(frame, face_mesh):
    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = face_mesh.process(rgb)
    if not results.multi_face_landmarks:
        return None
    pts = results.multi_face_landmarks[0].landmark
    return pts

### Optical Flow

In [9]:
def compute_optical_flow(prev_gray, gray):
    flow = cv2.calcOpticalFlowFarneback(prev_gray, gray, None,
                                        pyr_scale=0.5, levels=3, winsize=15,
                                        iterations=3, poly_n=5, poly_sigma=1.1, flags=0)
    return {
        "flow_mean_x": float(flow[...,0].mean()),
        "flow_mean_y": float(flow[...,1].mean()),
        "flow_std_x": float(flow[...,0].std()),
        "flow_std_y": float(flow[...,1].std())
    }

### Head pose

In [10]:
def calculate_head_pose(landmarks, frame_width, frame_height):
        model_points = np.array([
            (0.0, 0.0, 0.0),             # Nose tip
            (0.0, -330.0, -65.0),        # Chin
            (-225.0, 170.0, -135.0),     # Left eye left corner
            (225.0, 170.0, -135.0),      # Right eye right corner
            (-150.0, -150.0, -125.0),    # Left Mouth corner
            (150.0, -150.0, -125.0)      # Right mouth corner
        ])

        image_points = []
        for idx in [1, 152, 263, 33, 291, 61]:
            lm = landmarks[idx]
            x, y = lm.x * frame_width, lm.y * frame_height
            image_points.append([x, y])
            
        image_points = np.array(image_points, dtype="double")

        focal_length = frame_width
        center = (frame_width / 2, frame_height / 2)
        camera_matrix = np.array(
            [[focal_length, 0, center[0]],
             [0, focal_length, center[1]],
             [0, 0, 1]], dtype="double"
        )
        dist_coeffs = np.zeros((4, 1)) 

        success, rotation_vector, translation_vector = cv2.solvePnP(
            model_points, image_points, camera_matrix, dist_coeffs, flags=cv2.SOLVEPNP_ITERATIVE
        )

        if not success:
            return {'head_pitch': 0.0, 'head_yaw': 0.0, 'head_roll': 0.0}

        rotation_matrix, _ = cv2.Rodrigues(rotation_vector)
        proj_matrix = np.hstack((rotation_matrix, translation_vector))
        euler_angles = cv2.decomposeProjectionMatrix(proj_matrix)[6]
        
        return {
            'head_pitch': float(euler_angles[0].item()),
            'head_yaw': float(euler_angles[1].item()),
            'head_roll': float(euler_angles[2].item())
        }

### All together

In [11]:
def process_segment(video_cap, start_frame, end_frame, label, sample_id, face_detector, emotion_detector, face_mesh, frame_skip):
    results = []
    
    video_cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)
    
    current_frame = start_frame
    processed_count = 0
    prev_gray = None

    vid_height = int(video_cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    vid_width = int(video_cap.get(cv2.CAP_PROP_FRAME_WIDTH))

    while current_frame <= end_frame:
        ret, frame = video_cap.read()
        if not ret:
            break

        if processed_count % frame_skip != 0:
            current_frame += 1
            processed_count += 1
            continue

        face, box = face_crop(face_detector, frame)

        if face is None:
            current_frame += 1
            processed_count += 1
            continue

        x1, y1, x2, y2 = box
        box_center_x = (x1 + x2) / 2 / vid_width
        box_center_y = (y1 + y2) / 2 / vid_height
        box_width = (x2 - x1) / vid_width

        resized_face = resize_frame(face)

        landmarks = extract_landmarks(resized_face, face_mesh)
        if landmarks is None:
            landmarks_flat = [0.0] * (478*2)
            head_pose = {'head_pitch': 0.0, 'head_yaw': 0.0, 'head_roll': 0.0}
        else: 
            landmarks_flat = np.array([(p.x, p.y) for p in landmarks], dtype=np.float32).flatten()
            head_pose = calculate_head_pose(landmarks, 224, 224)


        gray = cv2.cvtColor(resized_face, cv2.COLOR_BGR2GRAY)
        
        if prev_gray is not None:
            flow = compute_optical_flow(prev_gray, gray)
        else:
            flow = {"flow_mean_x": 0.0, "flow_mean_y": 0.0, "flow_std_x": 0.0, "flow_std_y": 0.0}
        prev_gray = gray

        normalized_face = geometric_normalization(resized_face, landmarks)
        emotions = detect_emotions(normalized_face, emotion_detector)

        results.append({
            'id': sample_id,
            'frame': current_frame,
            'deceptive': label,
            'box_center_x': box_center_x,
            'box_center_y': box_center_y,
            'box_width': box_width,
            **head_pose,
            **{f"lm_{i}": landmarks_flat[i] for i in range(len(landmarks_flat))},
            **emotions,
            **flow
        })

        current_frame += 1
        processed_count += 1

    return results

In [12]:
def process_video(sample_id, video_path, segmenter, 
                            face_detector, emotion_detector, face_mesh, frame_skip):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        logging.error(f"Could not open {video_path}")
        return sample_id, []

    segments = segmenter.get_segments(video_path)
    
    if not segments:
        cap.release()
        return sample_id, []

    logging.info(f"Processing {video_path}: Found {len(segments)} segments.")
    
    all_video_results = []
    
    for start, end, label in segments:
        segment_results = process_segment(
            cap, start, end, label, sample_id,
            face_detector, emotion_detector, face_mesh, frame_skip
        )
        
        if len(segment_results) > 0:
            all_video_results.extend(segment_results)
            sample_id += 1 

    cap.release()
    return sample_id, all_video_results

In [13]:
def process_dataset(
    root_dir, 
    out_path, 
    dataset_type='silesian', # 'silesian' or 'simple'
    labels_dict=None,        # Needed if type='simple'
    frame_skip=5, 
    device=device
):
    logging.info(f"Starting processing for {dataset_type} dataset...")

    face_detector = YOLO('../model_weights/yolov8n-face.pt').to(device)
    emotion_detector = EmotionRecognition(device='gpu' if device == 'cuda' else 'cpu')
    
    if dataset_type == 'silesian':
        segmenter = SilesianSegmenter()
    elif dataset_type == 'simple':
        if labels_dict is None:
            raise ValueError("labels_dict is required for 'simple' dataset type")
        segmenter = SimpleLabelSegmenter(labels_dict)
    else:
        raise ValueError(f"Unknown dataset type: {dataset_type}")

    sample_id = 0
    
    header_written = False
    if os.path.exists(out_path):
        os.remove(out_path)

    mp_face_mesh = mp.solutions.face_mesh
    with mp_face_mesh.FaceMesh(
        static_image_mode=False,
        refine_landmarks=True,
        max_num_faces=1
    ) as face_mesh:
        
        video_files = []
        for root, dirs, files in os.walk(root_dir):
            for file in files:
                if file.lower().endswith(('.avi', '.mp4', '.mov')):
                    video_files.append(os.path.join(root, file))

        for video_path in tqdm(video_files, desc="Processing Videos"):
            
            sample_id, results = process_video(
                sample_id, video_path, segmenter,
                face_detector, emotion_detector, face_mesh, frame_skip
            )
            
            if len(results) > 0:
                df = pd.DataFrame(results)
                df.to_csv(out_path, mode="a", index=False, header=not header_written)
                header_written = True

            gc.collect()
            torch.cuda.empty_cache()

    logging.info("Dataset processing complete!")

# Real Life Deception Detection

In [18]:
#TODO: rerun
labels_dict = create_labels_from_filenames('../data/real_life_deception_detection_dataset')
process_dataset(root_dir='../data/real_life_deception_detection_dataset', out_path='../processed_data/real_life_deception_detection_dataset/data30fps.csv', dataset_type='simple', labels_dict=labels_dict, frame_skip=3, device=device)

📂 Scanning ../data/real_life_deception_detection_dataset for labels...
✅ Found 121 labeled videos.
2025-12-10 14:00:58,042 [INFO] Starting processing for simple dataset...


I0000 00:00:1765371658.149384    7634 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1765371658.184988   12793 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 580.95.05), renderer: NVIDIA GeForce RTX 2060/PCIe/SSE2


[*] Accuracy: 0.9565809379727686


W0000 00:00:1765371658.186630   12788 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
Processing Videos:   0%|          | 0/121 [00:00<?, ?it/s]

2025-12-10 14:00:58,194 [INFO] Processing ../data/real_life_deception_detection_dataset/Test/trial_lie_057.mp4: Found 1 segments.


...

2025-12-10 14:14:31,605 [INFO] Processing ../data/real_life_deception_detection_dataset/Train/trial_truth_053.mp4: Found 1 segments.


Processing Videos:  99%|█████████▉| 120/121 [13:40<00:04,  4.54s/it]

2025-12-10 14:14:38,451 [INFO] Processing ../data/real_life_deception_detection_dataset/Train/trial_truth_054.mp4: Found 1 segments.


Processing Videos: 100%|██████████| 121/121 [13:46<00:00,  6.83s/it]

2025-12-10 14:14:44,633 [INFO] Dataset processing complete!


## Silesian Deception Dataset

In [16]:
process_dataset(root_dir='../data/silesian_deception_dataset', out_path='../processed_data/silesian_deception_dataset/data10fps.csv', dataset_type='silesian', frame_skip=10, device=device)

2025-12-10 12:51:35,346 [INFO] Starting processing for silesian dataset...


I0000 00:00:1765367495.453452    7634 gl_context_egl.cc:85] Successfully initialized EGL. Major : 1 Minor: 5
I0000 00:00:1765367495.488860    9714 gl_context.cc:357] GL version: 3.2 (OpenGL ES 3.2 NVIDIA 580.95.05), renderer: NVIDIA GeForce RTX 2060/PCIe/SSE2
W0000 00:00:1765367495.490095    9711 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


[*] Accuracy: 0.9565809379727686


Processing Videos:   0%|          | 0/101 [00:00<?, ?it/s]

2025-12-10 12:51:35,496 [INFO] Processing ../data/silesian_deception_dataset/poli2Video/person1.avi: Found 9 segments.


...

2025-12-10 13:42:05,157 [INFO] Processing ../data/silesian_deception_dataset/poli3Video/person8.avi: Found 10 segments.


Processing Videos:  99%|█████████▉| 100/101 [51:06<00:37, 37.03s/it]

2025-12-10 13:42:42,449 [INFO] Processing ../data/silesian_deception_dataset/poli3Video/person9.avi: Found 10 segments.


Processing Videos: 100%|██████████| 101/101 [51:36<00:00, 30.66s/it]

2025-12-10 13:43:12,505 [INFO] Dataset processing complete!
